### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos el archivo movie_genre.json de nuestro contenedor bronze

In [0]:
movie_genre_schema = "movieId INT, genreId INT"

In [0]:
df = spark.read \
    .schema(movie_genre_schema) \
    .json(f"{bronze_folder_path}/movie_genre.json")

###### Cambiamos el nombre de las columnas

In [0]:
from pyspark.sql.functions import col,current_timestamp, lit

df_renamed = df.withColumnRenamed("movieId", "movie_id")\
                        .withColumnRenamed("genreId", "genre_id")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
df_add = add_columnas_control(df_renamed,v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_genre")